# Phase II validation notebook
**Agentic AI Multi-Agent Decision Support Platform for Industrial IoT Predictive Maintenance**
M.Tech Phase II, PES University · Sumanth GM (PES2PG24DS312) · Guide: Dr. Pulkit Dwivedi

This notebook shows every result of the project and can re-run each experiment. It works in **Google Colab** and in **VS Code** (Jupyter extension).

| Exp. | Question | Data | Folder |
| --- | --- | --- | --- |
| E1 | Can one model predict 19 inverter and pump-drive alarms before they trip? | Twin, simulated | `results/synthetic_weather` |
| E2 | Does it work on unseen 3 / 10 kW and 3 / 10 HP units with only a profile change? | Twin, simulated | `results/synthetic_weather` |
| E3 | How accurately does the Forecast Agent predict minutes to trip? | Twin, simulated | `results/synthetic_weather` |
| E4 | Does the Advisor return the right fix entry? | Twin, simulated | `results/synthetic_weather` |
| E5 | Do the same features + XGBoost recognise **real** inverter-drive faults? | **Real** PMSM lab data | `results/real_faults` |
| E6 | Do E1–E2 hold with **real** measured weather? | Twin + **real** La Reunion weather | `results/real_weather` |

By default the notebook **loads the saved results** (fast). Set the switches in section 1 to re-run any experiment.

## 1. Setup

In [ ]:
import os, sys, pathlib, subprocess, zipfile
IN_COLAB = "google.colab" in sys.modules
GIT_URL = ""   # e.g. "https://github.com/<you>/pdm_agentic.git"  (leave empty to upload pdm_agentic.zip)

if IN_COLAB:
    if GIT_URL:
        subprocess.run(["git", "clone", GIT_URL, "/content/repo"], check=True)
        ROOT = pathlib.Path("/content/repo")
        ROOT = ROOT / "pdm_agentic" if (ROOT / "pdm_agentic" / "src").exists() else ROOT
    else:
        if not pathlib.Path("/content/pdm_agentic/src").exists():
            from google.colab import files
            uploaded = files.upload()                      # choose pdm_agentic.zip
            zipfile.ZipFile(next(iter(uploaded))).extractall("/content")
        ROOT = pathlib.Path("/content/pdm_agentic")
else:
    here = pathlib.Path.cwd()
    ROOT = here.parent if here.name == "notebooks" else here

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Project folder:", ROOT)

In [ ]:
# ---- Switches -------------------------------------------------------------
RUN_E1_E4 = False   # True: re-run E1-E4 (QUICK about 2 min; full about 30 min)
QUICK     = True    # used when RUN_E1_E4 is True
RUN_E5    = False   # True: download the 1 MB PMSM dataset and re-run the real-fault check (about 1 min)
RUN_E6    = False   # True: stream the 2.2 GB La Reunion file (about 6 min) and re-run E1-E2 with real weather (about 6 min)
# ---------------------------------------------------------------------------
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost", "pyyaml", "fastapi", "httpx"], check=True)
import pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option("display.precision", 3)
from pdm.eval.report import METHOD_SHORT
RES = ROOT / "results"

In [ ]:
def load_results(folder):
    """Read the tables written by run_experiments.py from a results folder."""
    f = pathlib.Path(folder)
    read = lambda n: pd.read_csv(f / n) if (f / n).exists() else None
    return dict(summary=read("e1_e2_summary.csv"), per_alarm=read("e1_e2_per_alarm.csv"),
                forecast=read("e3_forecast_mae.csv"), time_to_trip=read("e3_time_to_trip.csv"),
                advisor=read("e4_advisor.csv"), out=f)

def e2_table(summary):
    """Macro-F1 per method and rating, plus the drop from the trained rating."""
    s = summary.copy(); s["method"] = s["method"].map(METHOD_SHORT).fillna(s["method"])
    s["size"] = s["rating"].str.extract(r"_(\d+)(?:kw|hp)$")[0].astype(int)
    t = s.pivot_table(index=["asset", "method"], columns="size", values="macro_f1")
    t["drop (worst unseen)"] = t[5] - t[[3, 10]].min(axis=1)
    return t

## 2. E1–E4 · Simulated telemetry twin

In [ ]:
if RUN_E1_E4:
    from pdm.experiments import run_all
    res = run_all(quick=QUICK)
else:
    res = load_results(RES / "synthetic_weather")
OUT = res["out"]; print("Results from:", OUT)

### E1 · Early warning on the trained rating (5 kW / 5 HP)

In [ ]:
s = res["summary"].copy(); s["method"] = s["method"].map(METHOD_SHORT).fillna(s["method"])
s[s.trained_on][["asset", "method", "macro_f1", "macro_pr_auc", "event_recall", "median_lead_min", "false_alerts_per_day"]]

### E2 · Unseen power ratings (no retraining, profile swap only)

In [ ]:
display(e2_table(res["summary"]))
display(Image(filename=str(OUT / "fig_cross_rating.png")))

### Per-alarm results, lead time and one worked example

In [ ]:
pa = res["per_alarm"]
(pa[(pa.method == "ours_pu_xgb") & pa.rating.isin(["inverter_hybrid_5kw", "pump_drive_5hp"])]
   .groupby(["asset", "alarm"])[["precision", "recall", "f1", "event_recall", "median_lead_min"]].mean())

In [ ]:
display(Image(filename=str(OUT / "fig_lead_time.png")))
display(Image(filename=str(OUT / "fig_example_INV-11.png")))

### E3 · Forecast Agent and E4 · Advisor Agent

In [ ]:
display(res["forecast"].pivot_table(index=["asset", "signal"], columns="method", values="mae_pu"))
tt = res["time_to_trip"].copy(); tt["caught"] = tt["caught"].astype(bool)
display(tt.groupby(["asset", "alarm", "method"]).agg(events=("caught", "size"), caught=("caught", "sum"),
                                                    mean_error_min=("abs_err_min", "mean")))
adv = res["advisor"]
adv.groupby("asset")[["ours_hit1", "ours_hit3", "readme_rag_hit1", "readme_rag_hit3"]].mean()

## 3. E5 · Real faults: inverter-driven PMSM lab dataset
**Data.** Zenodo 10.5281/zenodo.14482932 (CC-BY-4.0). A MOSFET inverter driving a PMSM, 10 samples/s, 8 signals, 9 classes:
F0 normal, F1–F2 switch open-circuit, F3–F5 switch short-circuit, F6–F8 half-bridge overheating.

**Link to the platform.** These classes are component faults, not the platform's 19 alarm codes, so the saved models are not used here. The **same window features and XGBoost settings** are re-trained on the real signals. This checks the method on real measurements.

**Split.** Each class is one recording, so every recording is cut into 4 time blocks; each fold tests one block and trains only on windows that share no samples with it.
The fair test uses the electrical signals only (Ia, Ib, VDC, IDC, VD) and classes F0–F5, because temperatures differ between recordings even without a fault.

In [ ]:
if RUN_E5:
    subprocess.run([sys.executable, "scripts/prepare_pmsm.py"], check=True)
    common = ["--csv", "data/real/pmsm_10hz.csv", "--split", "blocks", "--window", "20", "--out", "results/real_faults"]
    elec = ["--signals", "Ia", "Ib", "VDC", "IDC", "VD", "--labels", "F0", "F1", "F2", "F3", "F4", "F5"]
    for extra in (["--tag", "window"], ["--features", "instant", "--tag", "instant"],
                  elec + ["--tag", "electrical_window"], elec + ["--features", "instant", "--tag", "electrical_instant"]):
        subprocess.run([sys.executable, "scripts/validate_real.py", *common, *extra], check=True)

import json
rows = []
for f in sorted((RES / "real_faults").glob("*.json")):
    r = json.loads(f.read_text())
    rows.append({"test": r["tag"], "classes": len(r["classes"]), "signals": len(r["signals"]),
                 "features": r["features"], "test_windows": r["test_windows"],
                 "macro_f1": r["macro_f1"], "accuracy": r["accuracy"], "fault_detection_f1": r.get("detection_f1")})
pd.DataFrame(rows).set_index("test")

In [ ]:
import matplotlib.pyplot as plt
cm = pd.read_csv(RES / "real_faults" / "real_pmsm_10hz_electrical_window_confusion.csv", index_col=0)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm.values, cmap="Blues")
ax.set_xticks(range(len(cm.columns)), cm.columns); ax.set_yticks(range(len(cm.index)), cm.index)
for i in range(len(cm)):
    for j in range(len(cm.columns)):
        ax.text(j, i, cm.values[i, j], ha="center", va="center", fontsize=8,
                color="white" if cm.values[i, j] > cm.values.max() / 2 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("E5 real switch faults, electrical signals, 2 s windows")
plt.tight_layout(); plt.show()

## 4. E6 · Real measured weather (La Reunion rooftop PV)
**Data.** Zenodo 10.5281/zenodo.7157424 (CC-BY-4.0): plane-of-array irradiance and ambient temperature, Oct 2021 – Apr 2023, reduced to 1-minute values (519 complete days).
The twin uses this weather instead of synthetic weather; device telemetry and faults are still simulated. Training uses days 0–59 and each test seed its own later block, so test weather is never seen in training.

In [ ]:
if RUN_E6:
    if not (ROOT / "data/real/reunion_weather_1min.csv").exists():
        subprocess.run([sys.executable, "scripts/prepare_reunion_weather.py"], check=True)
    from pdm.experiments import run_all
    res6 = run_all(quick=False, out_dir=RES / "real_weather", weather="data/real/reunion_weather_1min.csv")
else:
    res6 = load_results(RES / "real_weather")
display(e2_table(res6["summary"]))
display(Image(filename=str(res6["out"] / "fig_cross_rating.png")))

In [ ]:
# Side by side: synthetic weather (E1-E2) vs measured weather (E6), our method and the ablation
both = []
for name, r in (("synthetic weather", res), ("real weather", res6)):
    t = r["summary"][r["summary"].method.isin(["ours_pu_xgb", "ablation_raw_xgb"])].copy()
    t["weather"] = name; both.append(t)
both = pd.concat(both); both["method"] = both["method"].map(METHOD_SHORT)
both.pivot_table(index=["asset", "method", "trained_on"], columns="weather", values="macro_f1")

## 5. Live-style demo: the 5 kW model on a 10 kW inverter (profile swap only)

In [ ]:
from pdm.api_helpers import platform_for
from pdm.twin import generate
from pdm.agents.advisor import format_checklist
plat = platform_for("inverter_hybrid_10kw", model_dir=RES / "models")   # the saved live models
stream = generate("inverter_hybrid_10kw", days=2, seed=2026, n_true=3, n_near=1)["raw"]
last, shown = None, 0
for t in range(60, len(stream), 5):
    out = plat.step(stream.iloc[max(0, t - 120):t])
    if out["state"] != last and out["state"] != "NORMAL":
        items = out["alarms"] or out["warnings"]
        print(out["time"], out["state"], [(i["code"], i["name"]) for i in items])
        if out["advice"] and shown < 2:
            print(format_checklist(out["advice"])); print(); shown += 1
    last = out["state"]

## 6. Summary
| Claim | Evidence |
| --- | --- |
| One platform predicts many alarms early | E1: 19 alarms, macro-F1 0.645 (inverter) and 0.572 (pump drive), median lead about 24–28 min |
| One model works across power ratings | E2 and E6: per-unit model changes by at most 0.02 on unseen ratings; raw inputs lose up to 0.20 |
| Better than the ASPIRE-style baseline | E1: Transformer reaches 0.385 / 0.215 with more false alerts |
| Minutes to trip | E3: trend method caught 10 of 12 thermal trips 30 min ahead |
| Advice tied to the prediction | E4: right entry first for about 95% of events, against about 10% for README-style RAG |
| Method works on real faults | E5: macro-F1 0.960 on real switch faults (electrical signals, 2 s windows), 0.632 without windows |

**Limitations.** The 19 alarm faults are simulated; E5 uses real faults with a different label set and tests classification, not early warning; E6 uses real weather with simulated device behaviour. Profile limits and knowledge-base sources are placeholders until replaced with a specific device's datasheet and manual.